In [0]:
%sql
-- CREATE TABLE silver_coingecko_market AS
-- SELECT 
--     id,
--     symbol,
--     name,
--     CAST(current_price AS DOUBLE) AS current_price_usd,
--     CAST(market_cap AS DOUBLE) AS market_cap_usd,
--     CAST(total_volume AS DOUBLE) as total_volume_usd,
--     CAST(high_24h AS DOUBLE) AS high_24h_usd,
--     CAST(low_24h AS DOUBLE) AS low_24h_usd,
--     CAST(price_change_24h AS DOUBLE) AS price_change_24h_usd,
--     CAST(price_change_percentage_24h AS DOUBLE) AS price_change_pct_24h,
--     CAST(market_cap_rank AS INT) AS market_cap_rank,
--     TO_TIMESTAMP(last_updated) AS last_updated_at
-- FROM bronze_coingecko_market
-- WHERE CAST(current_price AS DOUBLE) > 0;
-- 1. Creamos la tabla si no existe (con tus columnas exactas + las nuevas de fecha)

CREATE TABLE IF NOT EXISTS silver_coingecko_market (
    id STRING, 
    symbol STRING, 
    name STRING, 
    current_price_usd DOUBLE, 
    market_cap_usd DOUBLE, 
    total_volume_usd DOUBLE, 
    high_24h_usd DOUBLE, 
    low_24h_usd DOUBLE, 
    price_change_24h_usd DOUBLE, 
    price_change_pct_24h DOUBLE, 
    market_cap_rank INT, 
    last_updated_at TIMESTAMP, 
    ingestion_timestamp TIMESTAMP,
    ingestion_date DATE
) USING DELTA;

-- 2. Preparamos los datos nuevos (casteo y limpieza, respetando tu SELECT)
WITH nuevos_datos AS (
    SELECT 
        id,
        symbol,
        name,
        TRY_CAST(current_price AS DOUBLE) AS current_price_usd,
        TRY_CAST(market_cap AS DOUBLE) AS market_cap_usd,
        TRY_CAST(total_volume AS DOUBLE) as total_volume_usd,
        TRY_CAST(high_24h AS DOUBLE) AS high_24h_usd,
        TRY_CAST(low_24h AS DOUBLE) AS low_24h_usd,
        TRY_CAST(price_change_24h AS DOUBLE) AS price_change_24h_usd,
        TRY_CAST(price_change_percentage_24h AS DOUBLE) AS price_change_pct_24h,
        TRY_CAST(market_cap_rank AS INT) AS market_cap_rank,
        TO_TIMESTAMP(last_updated) AS last_updated_at,
        ingestion_timestamp,
        TRY_CAST(ingestion_timestamp AS DATE) AS ingestion_date
    FROM bronze_coingecko_market
    WHERE TRY_CAST(current_price AS DOUBLE) > 0
)

-- 3. Hacemos el MERGE (Upsert): Actualiza si ya existe hoy, inserta si es nuevo
MERGE INTO silver_coingecko_market AS target
USING nuevos_datos AS source
ON target.id = source.id AND target.ingestion_date = source.ingestion_date
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;